In [2]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..","DTENV"))
Pkg.instantiate()
include("../scripts/TesselationCore.jl")
if size(LOAD_PATH,1) < 4
    push!(LOAD_PATH, joinpath(@__DIR__,"..","scripts"))
end


  Activating project at `c:\Users\Ivan\Desktop\Stuff4School\Thesis\CleanDTFE\DTENV`


4-element Vector{String}:
 "@"
 "@v#.#"
 "@stdlib"
 "c:\\Users\\Ivan\\Desktop\\Stuff4School\\Thesis\\CleanDTFE\\notebooks\\..\\scripts"

In [3]:
using TetGen
using StaticArrays
using GLMakie
using JLD
using BenchmarkTools
using LinearAlgebra
using Plots

import .TesselationCore


In [54]:
struct BVHTree
    depth::Int
    rightChild::Union{Nothing,BVHTree}
    leftChild::Union{Nothing,BVHTree} #nothing if leaf, tree if node
    data::Vector{Int64}  # simplex index
end


struct BVH
    tree::BVHTree
    bbox::Matrix{Float64}   # global bounding box saved only once
end


function BVHTree(boxes,depth::Int,limBox::Matrix)
    #read whiteboard code

    indices = size(boxes,1)

    return BVHTree(boxes,depth,limBox,indices)
    
end


function BVHTree(boxes,depth::Int,limBox::Matrix, indices)
    #at each depth step, gets dimension (x,y,z one after another), finds halway point, creates left and right
    
    if depth == 0
        return BVHTree(0,nothing,nothing,indices)
    end
    
    ax = depth%3 + 1

    mins = boxes[ax,1,:]
    maxs = boxes[ax,1,:]
    
    line = limBox[ax,2]-limBox[ax,1]

    ids = size(boxes,1)
    
    leftIDs = ids[min.≤line]
    rightIDs = ids[max.≥line]

    leftBox = copy(limBox)
    leftBox[ax,2] = line
    
    rightBox = copy(limBox)
    rightBox[ax,1] = line


    return BVHTree(depth,
    BVHTree(boxes[leftIDs],depth-1,leftBox,indices[leftIDs]),
    BVHTree(boxes[rightIDs],depth-1,rightBox,indices[rightIDs]),
    indices)

end

function BVH(data::Vector,depth::Int)
    boxes = stack([cornerSimplexMatr(simplex) for simplex in data])

    minima = (minimum(boxes[1,1,:]),minimum(boxes[2,1,:]),minimum(boxes[3,1,:]))
    maxima = (maximum(boxes[1,2,:]),maximum(boxes[2,2,:]),maximum(boxes[3,2,:]))

    box = stack([minima,maxima])


    tree = BVHTree(boxes,depth,box)
    

    # generate bounding box; simplify data to just corners
    # call BVHtree to get the tree -> create BVH object
    return BVH(tree,box)
end


function BVH(data::Vector,depth::Int,box::Matrix)
    boxes = stack([cornerSimplexMatr(simplex) for simplex in data])

    tree = BVHTree(boxes,depth,box)
    return BVH(tree,box)
end

BVH(simplices,1)

MethodError: MethodError: no method matching isless(::typeof(min), ::Float64)
The function `isless` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  isless(!Matched::Missing, ::Any)
   @ Base missing.jl:87
  isless(::Any, !Matched::Missing)
   @ Base missing.jl:88
  isless(!Matched::Unitful.AbstractQuantity, ::Number)
   @ Unitful C:\Users\Ivan\.julia\packages\Unitful\HCJFR\src\quantities.jl:238
  ...


In [5]:
points3d = [TesselationCore.point3(@SVector rand(3)) for _ in 1:10]
coords, tets = TesselationCore.tesselate(points3d)
simplices = Vector([coords[:,tets[i,:]]' for i in 1:size(tets,1)])
simplex = simplices[1]


4×3 adjoint(::Matrix{Float64}) with eltype Float64:
 0.244961  0.619506  0.169562
 0.100209  0.887922  0.909653
 0.675962  0.824573  0.576
 0.788158  0.992794  0.080996

In [ ]:
function cornerSimplex(simplex)
    box = [minimum(simplex,dims=1),maximum(simplex,dims=1)]
    return box
end


function cornerSimplexMatr(simplex)
    box = hcat(minimum(simplex,dims=1)',maximum(simplex,dims=1)') 
    return box
end

cornerSimplexMatr (generic function with 1 method)

In [ ]:
@btime cornerSimplex(simplex)
@btime cornerSimplexMatr(simplex) # same speed lmao

  3.337 μs (22 allocations: 720 bytes)
  3.350 μs (22 allocations: 768 bytes)


3×2 Matrix{Float64}:
 0.100209  0.788158
 0.619506  0.992794
 0.080996  0.909653